# 09 — Data Inventory
## Inventaire complet de toutes les données sur GCS

Résumé de l'ensemble du pipeline : couverture temporelle, volumes, coûts estimés.

In [ ]:
import os, sys, json
sys.path.insert(0, "/workspace")

from btc_pipeline.storage.gcs_client import StorageClient

os.environ.setdefault("GCS_BUCKET_NAME", "btc-training-data")
os.environ.setdefault("STORAGE_BACKEND", "gcs")
os.environ.setdefault("WORKSPACE_DIR",  "/workspace")

storage = StorageClient()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Inventaire complet
# ════════════════════════════════════════════════════════════════════════
inventory = storage.get_inventory()

print("═" * 70)
print("INVENTAIRE GCS — BTC DATA PIPELINE")
print("═" * 70)

total_files = 0
total_gb = 0

for prefix, info in sorted(inventory.items()):
    n = info["file_count"]
    gb = info.get("total_size_gb", -1)
    total_files += n
    if gb > 0:
        total_gb += gb
    size_str = f"{gb:.2f} GB" if gb >= 0 else "N/A"
    print(f"  {prefix}")
    print(f"    Files: {n:,}  |  Size: {size_str}")
    if info.get("first_file"):
        print(f"    First: {info['first_file']}")
    if info.get("last_file"):
        print(f"    Last:  {info['last_file']}")
    print()

print("─" * 70)
print(f"TOTAL: {total_files:,} fichiers, {total_gb:.2f} GB")
print(f"Coût GCS estimé : ~${total_gb * 0.02:.2f}/mois (Standard storage)")
print("═" * 70)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Couverture temporelle
# ════════════════════════════════════════════════════════════════════════
import pandas as pd

print("\n📅 COUVERTURE TEMPORELLE")
print("─" * 70)

coverage = {}

# Check aggTrades coverage
agg_files = sorted(storage.list_files("raw/binance/spot_aggtrades/"))
if agg_files:
    # Extract year-month from filenames
    months = set()
    for f in agg_files:
        parts = f.split("/")
        for p in parts:
            if p.startswith("year="):
                year = p.replace("year=", "")
            if p.startswith("month="):
                month = p.replace("month=", "")
                months.add(f"{year}-{month}")
    if months:
        coverage["aggTrades spot"] = f"{min(months)} → {max(months)} ({len(months)} mois)"

# Check features coverage
feat_files = sorted(storage.list_files("processed/features_1s/"))
if feat_files:
    months = set()
    for f in feat_files:
        parts = f.split("/")
        for p in parts:
            if p.startswith("year="):
                year = p.replace("year=", "")
            if p.startswith("month="):
                month = p.replace("month=", "")
                months.add(f"{year}-{month}")
    if months:
        coverage["Dataset features"] = f"{min(months)} → {max(months)} ({len(months)} mois)"

# Block coverage
block_files = sorted(storage.list_files("raw/blockchain/blocks/"))
if block_files:
    coverage["Blockchain blocs"] = f"{len(block_files)} fichiers"

for name, info in coverage.items():
    print(f"  {name}: {info}")

print("─" * 70)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Pipeline state final
# ════════════════════════════════════════════════════════════════════════
state = storage.get_pipeline_state()
print("\n📋 PIPELINE STATE")
print(json.dumps(state, indent=2, default=str))

# Sauvegarder l'inventaire
inv_json = json.dumps(inventory, indent=2, default=str).encode("utf-8")
storage.stream_upload_bytes(inv_json, "metadata/data_inventory.json", skip_if_exists=False)
print("\n✅ Inventaire sauvegardé sur GCS : metadata/data_inventory.json")